# K-Nearest Neighbors (KNN) - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.neighbors import KNeighborsClassifier as SklearnKNN
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is K-Nearest Neighbors?

K-Nearest Neighbors (KNN) is a **non-parametric, instance-based (lazy) learning** algorithm used for both classification and regression. It makes predictions based on the K closest training examples in the feature space.

### The KNN Algorithm

1. Store all training data (no explicit training phase)
2. For a new data point:
   - Calculate distance to all training points
   - Find the K nearest neighbors
   - For classification: majority vote among K neighbors
   - For regression: average of K neighbors' values

### Distance Metrics

#### Euclidean Distance (L2 Norm)
Most common distance metric, works well in low-dimensional spaces:
$$d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$

#### Manhattan Distance (L1 Norm)
Also called city-block or taxicab distance, better for high-dimensional data:
$$d(x, y) = \sum_{i=1}^{n} |x_i - y_i|$$

#### Minkowski Distance (Generalized)
Generalization of both Euclidean and Manhattan:
$$d(x, y) = \left( \sum_{i=1}^{n} |x_i - y_i|^p \right)^{1/p}$$

- When p=1: Manhattan distance
- When p=2: Euclidean distance
- When p=infinity: Chebyshev distance

### Voting Schemes

#### Uniform Voting (Standard)
Each neighbor has equal weight:
$$\hat{y} = \text{mode}(y_1, y_2, ..., y_K)$$

#### Distance-Weighted Voting
Closer neighbors have more influence:
$$w_i = \frac{1}{d(x, x_i) + \epsilon}$$
$$\hat{y} = \arg\max_c \sum_{i: y_i = c} w_i$$

### Choosing K

- **Small K** (e.g., K=1): High variance, low bias, sensitive to noise
- **Large K**: Low variance, high bias, smoother decision boundaries
- **Rule of thumb**: K = sqrt(n) where n is the number of training samples
- **Best practice**: Use cross-validation to find optimal K
- **Tip**: Use odd K for binary classification to avoid ties

### Time & Space Complexity (Lazy Learner)

KNN is a **lazy learner** - it does no work during training and defers computation to prediction time.

| Operation | Time Complexity | Space Complexity |
|-----------|-----------------|------------------|
| Training (fit) | O(1) | O(n * d) |
| Prediction (per sample) | O(n * d) | O(n) |
| Prediction (m samples) | O(m * n * d) | O(n) |

Where: n = training samples, d = features, m = test samples

**Note**: Prediction can be optimized to O(n * d + K * log(n)) using KD-trees or Ball trees.

## 2. Implementation from Scratch <a id='implementation'></a>

In [ ]:
class KNNClassifier:
    """
    K-Nearest Neighbors Classifier implementation from scratch.
    
    Parameters:
    -----------
    n_neighbors : int, default=5
        Number of neighbors to use for prediction
    metric : str, default='euclidean'
        Distance metric: 'euclidean', 'manhattan', or 'minkowski'
    p : int, default=2
        Power parameter for Minkowski distance
    weights : str, default='uniform'
        Weight function: 'uniform' (all equal) or 'distance' (inverse distance)
    """
    
    def __init__(self, n_neighbors=5, metric='euclidean', p=2, weights='uniform'):
        self.n_neighbors = n_neighbors
        self.metric = metric
        self.p = p
        self.weights = weights
        self.X_train = None
        self.y_train = None
        self.classes_ = None
        
    def _euclidean_distance(self, x1, x2):
        """
        Compute Euclidean distance between two points.
        Vectorized for efficiency when x2 is a matrix.
        """
        return np.sqrt(np.sum((x1 - x2) ** 2, axis=-1))
    
    def _manhattan_distance(self, x1, x2):
        """
        Compute Manhattan (L1) distance between two points.
        """
        return np.sum(np.abs(x1 - x2), axis=-1)
    
    def _minkowski_distance(self, x1, x2, p):
        """
        Compute Minkowski distance between two points.
        """
        return np.sum(np.abs(x1 - x2) ** p, axis=-1) ** (1 / p)
    
    def _compute_distances(self, x):
        """
        Compute distances from point x to all training points.
        Uses vectorized operations for efficiency.
        
        Parameters:
        -----------
        x : array-like, shape (n_features,)
            Query point
            
        Returns:
        --------
        distances : array, shape (n_train_samples,)
            Distances to all training points
        """
        if self.metric == 'euclidean':
            return self._euclidean_distance(x, self.X_train)
        elif self.metric == 'manhattan':
            return self._manhattan_distance(x, self.X_train)
        elif self.metric == 'minkowski':
            return self._minkowski_distance(x, self.X_train, self.p)
        else:
            raise ValueError(f"Unknown metric: {self.metric}")
    
    def fit(self, X, y):
        """
        Store training data (lazy learning - no actual training).
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Training data
        y : array-like, shape (n_samples,)
            Target labels
            
        Returns:
        --------
        self : object
        """
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        self.classes_ = np.unique(y)
        return self
    
    def _predict_single(self, x):
        """
        Predict class label for a single sample.
        
        Parameters:
        -----------
        x : array-like, shape (n_features,)
            Query point
            
        Returns:
        --------
        label : int or str
            Predicted class label
        """
        # Compute distances to all training points
        distances = self._compute_distances(x)
        
        # Get indices of K nearest neighbors
        k_indices = np.argsort(distances)[:self.n_neighbors]
        k_distances = distances[k_indices]
        k_labels = self.y_train[k_indices]
        
        # Voting
        if self.weights == 'uniform':
            # Simple majority vote
            vote_counts = Counter(k_labels)
            return vote_counts.most_common(1)[0][0]
        elif self.weights == 'distance':
            # Distance-weighted voting
            # Add small epsilon to avoid division by zero
            weights = 1.0 / (k_distances + 1e-10)
            
            # Weighted vote for each class
            class_weights = {}
            for label, weight in zip(k_labels, weights):
                class_weights[label] = class_weights.get(label, 0) + weight
            
            return max(class_weights, key=class_weights.get)
        else:
            raise ValueError(f"Unknown weights: {self.weights}")
    
    def predict(self, X):
        """
        Predict class labels for samples in X.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Query points
            
        Returns:
        --------
        labels : array, shape (n_samples,)
            Predicted class labels
        """
        X = np.array(X)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        return np.array([self._predict_single(x) for x in X])
    
    def predict_proba(self, X):
        """
        Predict class probabilities for samples in X.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Query points
            
        Returns:
        --------
        probas : array, shape (n_samples, n_classes)
            Class probabilities
        """
        X = np.array(X)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        
        probas = []
        for x in X:
            distances = self._compute_distances(x)
            k_indices = np.argsort(distances)[:self.n_neighbors]
            k_distances = distances[k_indices]
            k_labels = self.y_train[k_indices]
            
            if self.weights == 'uniform':
                # Count occurrences of each class
                proba = np.zeros(len(self.classes_))
                for i, cls in enumerate(self.classes_):
                    proba[i] = np.sum(k_labels == cls) / self.n_neighbors
            else:
                # Distance-weighted probabilities
                weights = 1.0 / (k_distances + 1e-10)
                proba = np.zeros(len(self.classes_))
                for i, cls in enumerate(self.classes_):
                    proba[i] = np.sum(weights[k_labels == cls])
                proba /= proba.sum()  # Normalize
            
            probas.append(proba)
        
        return np.array(probas)
    
    def score(self, X, y):
        """
        Return accuracy score.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Test data
        y : array-like, shape (n_samples,)
            True labels
            
        Returns:
        --------
        score : float
            Accuracy score
        """
        predictions = self.predict(X)
        return np.mean(predictions == y)

### Testing Our Implementation

In [ ]:
# Quick test with simple data
X_simple = np.array([[1, 1], [1, 2], [2, 1], [5, 5], [5, 6], [6, 5]])
y_simple = np.array([0, 0, 0, 1, 1, 1])

knn_test = KNNClassifier(n_neighbors=3)
knn_test.fit(X_simple, y_simple)

# Test predictions
test_points = np.array([[1.5, 1.5], [5.5, 5.5], [3, 3]])
predictions = knn_test.predict(test_points)
probas = knn_test.predict_proba(test_points)

print("Test Points:")
for i, (point, pred, prob) in enumerate(zip(test_points, predictions, probas)):
    print(f"  Point {point}: Predicted class = {pred}, Probabilities = {prob}")

# Visualize
plt.figure(figsize=(8, 6))
plt.scatter(X_simple[y_simple == 0, 0], X_simple[y_simple == 0, 1], 
            c='blue', label='Class 0', s=100, marker='o')
plt.scatter(X_simple[y_simple == 1, 0], X_simple[y_simple == 1, 1], 
            c='red', label='Class 1', s=100, marker='o')
plt.scatter(test_points[:, 0], test_points[:, 1], 
            c='green', label='Test Points', s=150, marker='*')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('KNN Simple Test')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Training & Optimization <a id='training'></a>

In [ ]:
# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features (important for KNN!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset: Iris")
print(f"Features: {feature_names}")
print(f"Classes: {target_names}")
print(f"\nTraining set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Class distribution in training: {np.bincount(y_train)}")

In [ ]:
# Train models with different configurations
models = {
    'K=3, Euclidean, Uniform': KNNClassifier(n_neighbors=3, metric='euclidean', weights='uniform'),
    'K=5, Euclidean, Uniform': KNNClassifier(n_neighbors=5, metric='euclidean', weights='uniform'),
    'K=5, Manhattan, Uniform': KNNClassifier(n_neighbors=5, metric='manhattan', weights='uniform'),
    'K=5, Euclidean, Distance': KNNClassifier(n_neighbors=5, metric='euclidean', weights='distance'),
    'K=7, Minkowski p=3': KNNClassifier(n_neighbors=7, metric='minkowski', p=3, weights='uniform'),
}

# Train and evaluate all models
print("Model Performance Comparison:")
print("=" * 60)
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    train_acc = model.score(X_train_scaled, y_train)
    test_acc = model.score(X_test_scaled, y_test)
    results.append({'Model': name, 'Train Acc': train_acc, 'Test Acc': test_acc})
    print(f"\n{name}:")
    print(f"  Train Accuracy: {train_acc:.4f}")
    print(f"  Test Accuracy: {test_acc:.4f}")

In [ ]:
# Find optimal K using cross-validation
from sklearn.model_selection import cross_val_score

def find_optimal_k(X, y, k_range=range(1, 31), cv=5):
    """
    Find optimal K using cross-validation.
    """
    cv_scores = []
    
    for k in k_range:
        knn = KNNClassifier(n_neighbors=k)
        # Manual cross-validation
        fold_size = len(X) // cv
        scores = []
        
        for i in range(cv):
            # Create fold indices
            val_start = i * fold_size
            val_end = (i + 1) * fold_size if i < cv - 1 else len(X)
            
            # Split data
            X_val = X[val_start:val_end]
            y_val = y[val_start:val_end]
            X_tr = np.concatenate([X[:val_start], X[val_end:]])
            y_tr = np.concatenate([y[:val_start], y[val_end:]])
            
            # Train and evaluate
            knn.fit(X_tr, y_tr)
            scores.append(knn.score(X_val, y_val))
        
        cv_scores.append(np.mean(scores))
    
    return list(k_range), cv_scores

# Find optimal K
k_values, cv_scores = find_optimal_k(X_train_scaled, y_train, k_range=range(1, 21))

# Plot K vs accuracy
plt.figure(figsize=(10, 6))
plt.plot(k_values, cv_scores, 'bo-', linewidth=2, markersize=8)
optimal_k = k_values[np.argmax(cv_scores)]
plt.axvline(x=optimal_k, color='r', linestyle='--', label=f'Optimal K = {optimal_k}')
plt.xlabel('K (Number of Neighbors)', fontsize=12)
plt.ylabel('Cross-Validation Accuracy', fontsize=12)
plt.title('Finding Optimal K via Cross-Validation', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(k_values)
plt.show()

print(f"\nOptimal K: {optimal_k} with CV accuracy: {max(cv_scores):.4f}")

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
# Train model with optimal K
best_knn = KNNClassifier(n_neighbors=optimal_k, metric='euclidean', weights='distance')
best_knn.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = best_knn.predict(X_train_scaled)
y_test_pred = best_knn.predict(X_test_scaled)

print(f"Best Model: K={optimal_k}, Distance-Weighted")
print(f"\nTrain Accuracy: {best_knn.score(X_train_scaled, y_train):.4f}")
print(f"Test Accuracy: {best_knn.score(X_test_scaled, y_test):.4f}")

# Classification Report
print("\n" + "="*60)
print("Classification Report (Test Set):")
print("="*60)
print(classification_report(y_test, y_test_pred, target_names=target_names))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training confusion matrix
cm_train = confusion_matrix(y_train, y_train_pred)
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names, ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix - Training Set')

# Test confusion matrix
cm_test = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names, ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix - Test Set')

plt.tight_layout()
plt.show()

In [ ]:
# Accuracy vs K plot (Training and Test)
k_range = range(1, 31)
train_accuracies = []
test_accuracies = []

for k in k_range:
    knn = KNNClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_accuracies.append(knn.score(X_train_scaled, y_train))
    test_accuracies.append(knn.score(X_test_scaled, y_test))

plt.figure(figsize=(12, 6))
plt.plot(k_range, train_accuracies, 'b-o', label='Training Accuracy', linewidth=2, markersize=6)
plt.plot(k_range, test_accuracies, 'r-s', label='Test Accuracy', linewidth=2, markersize=6)
plt.fill_between(k_range, train_accuracies, test_accuracies, alpha=0.2, color='gray')

# Mark optimal K
best_k_test = k_range[np.argmax(test_accuracies)]
plt.axvline(x=best_k_test, color='green', linestyle='--', 
            label=f'Best Test K = {best_k_test}')

plt.xlabel('K (Number of Neighbors)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Accuracy vs K - Bias-Variance Tradeoff', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.xticks(list(k_range)[::2])

# Add annotations
plt.annotate('Low K: High Variance\n(Overfitting)', xy=(2, 0.95), fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.annotate('High K: High Bias\n(Underfitting)', xy=(22, 0.85), fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.show()

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
def plot_decision_boundary_knn(X, y, model, title="KNN Decision Boundary", ax=None):
    """
    Plot decision boundary for 2D data.
    
    Parameters:
    -----------
    X : array-like, shape (n_samples, 2)
        Feature data (must be 2D)
    y : array-like, shape (n_samples,)
        Labels
    model : KNNClassifier
        Trained KNN model
    title : str
        Plot title
    ax : matplotlib axis, optional
        Axis to plot on
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    h = 0.02  # Step size in mesh
    
    # Create mesh
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.RdYlBu)
    ax.contour(xx, yy, Z, colors='black', linewidths=0.5)
    
    # Plot data points
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, 
                        edgecolor='black', s=50)
    
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title(title)
    
    return ax

In [ ]:
# Use only 2 features for visualization
X_2d = X_train_scaled[:, :2]  # Use first two features
y_2d = y_train

# Plot decision boundaries for different K values
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
k_values_viz = [1, 3, 5, 9, 15, 25]

for ax, k in zip(axes.flatten(), k_values_viz):
    knn = KNNClassifier(n_neighbors=k)
    knn.fit(X_2d, y_2d)
    plot_decision_boundary_knn(X_2d, y_2d, knn, f'K = {k}', ax=ax)

plt.suptitle('Effect of K on Decision Boundaries', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Compare distance metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['euclidean', 'manhattan', 'minkowski']
metric_titles = ['Euclidean (L2)', 'Manhattan (L1)', 'Minkowski (p=3)']

for ax, metric, title in zip(axes, metrics, metric_titles):
    if metric == 'minkowski':
        knn = KNNClassifier(n_neighbors=5, metric=metric, p=3)
    else:
        knn = KNNClassifier(n_neighbors=5, metric=metric)
    knn.fit(X_2d, y_2d)
    plot_decision_boundary_knn(X_2d, y_2d, knn, f'{title} Distance', ax=ax)

plt.suptitle('Distance Metric Comparison (K=5)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Compare uniform vs distance-weighted voting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Uniform weights
knn_uniform = KNNClassifier(n_neighbors=5, weights='uniform')
knn_uniform.fit(X_2d, y_2d)
plot_decision_boundary_knn(X_2d, y_2d, knn_uniform, 'Uniform Weights', ax=axes[0])

# Distance weights
knn_distance = KNNClassifier(n_neighbors=5, weights='distance')
knn_distance.fit(X_2d, y_2d)
plot_decision_boundary_knn(X_2d, y_2d, knn_distance, 'Distance-Weighted', ax=axes[1])

plt.suptitle('Voting Scheme Comparison (K=5)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize how KNN makes predictions for a single point
def visualize_knn_prediction(X, y, query_point, k, ax=None):
    """
    Visualize how KNN classifies a query point.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    # Compute distances
    distances = np.sqrt(np.sum((X - query_point) ** 2, axis=1))
    k_indices = np.argsort(distances)[:k]
    
    # Plot all points
    colors = ['blue', 'red', 'green']
    for i, cls in enumerate(np.unique(y)):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[i], 
                  label=f'Class {cls}', s=80, alpha=0.6)
    
    # Highlight K nearest neighbors
    ax.scatter(X[k_indices, 0], X[k_indices, 1], 
              facecolors='none', edgecolors='black', s=200, linewidth=3,
              label=f'{k} Nearest Neighbors')
    
    # Plot query point
    ax.scatter(query_point[0], query_point[1], c='yellow', 
              marker='*', s=400, edgecolor='black', linewidth=2,
              label='Query Point', zorder=5)
    
    # Draw lines to K nearest neighbors
    for idx in k_indices:
        ax.plot([query_point[0], X[idx, 0]], 
               [query_point[1], X[idx, 1]], 
               'k--', alpha=0.3)
    
    # Draw circle encompassing K nearest neighbors
    radius = distances[k_indices[-1]]
    circle = plt.Circle(query_point, radius, fill=False, 
                       color='gray', linestyle='--', linewidth=2)
    ax.add_patch(circle)
    
    # Prediction
    neighbor_labels = y[k_indices]
    prediction = Counter(neighbor_labels).most_common(1)[0][0]
    
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title(f'KNN Prediction (K={k})\nNeighbor labels: {list(neighbor_labels)} -> Prediction: Class {prediction}')
    ax.legend(loc='upper right')
    ax.set_aspect('equal')
    
    return ax

# Visualize prediction for a sample point
query_point = np.array([0.5, 0.5])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, k in zip(axes, [1, 5, 11]):
    visualize_knn_prediction(X_2d, y_2d, query_point, k, ax=ax)

plt.tight_layout()
plt.show()

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use KNN

#### Good Use Cases:

1. **Small to Medium Datasets**
   - Works well when n < 10,000 samples
   - Prediction time is acceptable

2. **Low-Dimensional Data**
   - Best with d < 20 features
   - Distance metrics are meaningful

3. **Interpretability Required**
   - Easy to explain: "classified based on similar examples"
   - Can show which neighbors influenced the decision

4. **Non-linear Decision Boundaries**
   - No assumption about data distribution
   - Can capture complex patterns

5. **Real-time Updates Needed**
   - New data can be added without retraining
   - Lazy learning advantage

6. **Multi-class Classification**
   - Naturally handles multiple classes
   - No need for one-vs-rest schemes

#### Example Applications:
- Recommendation systems
- Image classification (small datasets)
- Anomaly detection
- Pattern recognition
- Medical diagnosis (similar patient cases)

### When NOT to Use KNN

#### Avoid KNN When:

1. **Large Datasets**
   - Prediction time: O(n * d) per sample
   - Memory: Must store all training data
   - Use: Tree-based methods, Neural Networks

2. **High-Dimensional Data (Curse of Dimensionality)**
   - Distances become less meaningful
   - All points become "equally far"
   - Use: Dimensionality reduction first, or different algorithms

3. **Imbalanced Classes**
   - Majority class dominates neighborhoods
   - Use: Weighted voting, resampling, or different algorithms

4. **Noisy Data**
   - Sensitive to outliers and noise
   - Use: Noise filtering, larger K, or robust algorithms

5. **Real-time Predictions on Large Data**
   - Too slow for production systems
   - Use: Pre-computed indices (KD-trees), or parametric models

### The Curse of Dimensionality

As dimensions increase:
- Volume of space increases exponentially
- Data becomes sparse
- Distance between any two points converges
- "Nearest" neighbor may not be meaningful

In [ ]:
# Demonstrate the curse of dimensionality
def demonstrate_curse_of_dimensionality():
    """
    Show how distance ratios change with dimensionality.
    """
    np.random.seed(42)
    n_samples = 1000
    dimensions = [2, 5, 10, 20, 50, 100, 200, 500]
    
    distance_ratios = []
    
    for d in dimensions:
        # Generate random points in d-dimensional unit cube
        points = np.random.uniform(0, 1, (n_samples, d))
        query = np.random.uniform(0, 1, d)
        
        # Calculate distances
        distances = np.sqrt(np.sum((points - query) ** 2, axis=1))
        
        # Ratio of max to min distance
        ratio = distances.max() / distances.min()
        distance_ratios.append(ratio)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(dimensions, distance_ratios, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Number of Dimensions', fontsize=12)
    plt.ylabel('Max/Min Distance Ratio', fontsize=12)
    plt.title('Curse of Dimensionality: Distance Contrast Decreases', fontsize=14)
    plt.xscale('log')
    plt.grid(True, alpha=0.3)
    
    # Add annotation
    plt.annotate('As dimensions increase,\nall points become\n"equally far"', 
                xy=(100, distance_ratios[dimensions.index(100)]),
                xytext=(150, 3),
                fontsize=10,
                arrowprops=dict(arrowstyle='->', color='red'),
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.show()
    
    print("\nDistance Ratios by Dimension:")
    for d, ratio in zip(dimensions, distance_ratios):
        print(f"  d={d:3d}: Max/Min ratio = {ratio:.2f}")

demonstrate_curse_of_dimensionality()

In [ ]:
# Demonstrate importance of feature scaling
def demonstrate_scaling_importance():
    """
    Show the importance of feature scaling for KNN.
    """
    # Create dataset with different scales
    np.random.seed(42)
    n_samples = 200
    
    # Feature 1: range [0, 1], Feature 2: range [0, 1000]
    X = np.column_stack([
        np.random.uniform(0, 1, n_samples),
        np.random.uniform(0, 1000, n_samples)
    ])
    y = (X[:, 0] > 0.5).astype(int)  # True boundary based on Feature 1
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    # Without scaling
    knn_unscaled = KNNClassifier(n_neighbors=5)
    knn_unscaled.fit(X_train, y_train)
    acc_unscaled = knn_unscaled.score(X_test, y_test)
    
    # With scaling
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    knn_scaled = KNNClassifier(n_neighbors=5)
    knn_scaled.fit(X_train_s, y_train)
    acc_scaled = knn_scaled.score(X_test_s, y_test)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Unscaled
    axes[0].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdYlBu', 
                   edgecolor='black', s=50)
    axes[0].set_xlabel('Feature 1 (range 0-1)')
    axes[0].set_ylabel('Feature 2 (range 0-1000)')
    axes[0].set_title(f'Without Scaling\nAccuracy: {acc_unscaled:.2%}')
    axes[0].annotate('Feature 2 dominates\ndistance calculations!', 
                    xy=(0.5, 500), fontsize=10,
                    bbox=dict(boxstyle='round', facecolor='red', alpha=0.3))
    
    # Scaled
    axes[1].scatter(X_test_s[:, 0], X_test_s[:, 1], c=y_test, cmap='RdYlBu', 
                   edgecolor='black', s=50)
    axes[1].set_xlabel('Feature 1 (scaled)')
    axes[1].set_ylabel('Feature 2 (scaled)')
    axes[1].set_title(f'With Scaling\nAccuracy: {acc_scaled:.2%}')
    axes[1].annotate('Both features\ncontribute equally!', 
                    xy=(0, 0), fontsize=10,
                    bbox=dict(boxstyle='round', facecolor='green', alpha=0.3))
    
    plt.suptitle('Importance of Feature Scaling', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    print(f"\nAccuracy without scaling: {acc_unscaled:.2%}")
    print(f"Accuracy with scaling: {acc_scaled:.2%}")

demonstrate_scaling_importance()

### Pros and Cons Summary

| Pros | Cons |
|------|------|
| Simple and intuitive | Slow prediction (O(n)) |
| No training phase | High memory requirement |
| Naturally handles multi-class | Sensitive to irrelevant features |
| Non-parametric (no assumptions) | Curse of dimensionality |
| New data easily added | Requires feature scaling |
| Works with any distance metric | Sensitive to imbalanced data |
| Interpretable predictions | K must be chosen carefully |

### Best Practices

1. **Always scale features** (StandardScaler or MinMaxScaler)
2. **Use cross-validation** to find optimal K
3. **Try distance-weighted voting** for noisy data
4. **Reduce dimensionality** if d > 20 (PCA, feature selection)
5. **Use KD-trees** for faster queries on large datasets
6. **Handle imbalanced classes** with weighted voting

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare with sklearn implementation
from sklearn.neighbors import KNeighborsClassifier

# Our implementation
our_knn = KNNClassifier(n_neighbors=5, metric='euclidean', weights='uniform')
our_knn.fit(X_train_scaled, y_train)

# Sklearn implementation
sklearn_knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', weights='uniform')
sklearn_knn.fit(X_train_scaled, y_train)

# Predictions
our_pred = our_knn.predict(X_test_scaled)
sklearn_pred = sklearn_knn.predict(X_test_scaled)

# Probabilities
our_proba = our_knn.predict_proba(X_test_scaled)
sklearn_proba = sklearn_knn.predict_proba(X_test_scaled)

# Results
print("Performance Comparison:")
print("=" * 60)
print(f"\nOur Implementation:")
print(f"  Train Accuracy: {our_knn.score(X_train_scaled, y_train):.4f}")
print(f"  Test Accuracy: {our_knn.score(X_test_scaled, y_test):.4f}")
print(f"\nSklearn Implementation:")
print(f"  Train Accuracy: {sklearn_knn.score(X_train_scaled, y_train):.4f}")
print(f"  Test Accuracy: {sklearn_knn.score(X_test_scaled, y_test):.4f}")
print(f"\nPrediction Agreement: {np.mean(our_pred == sklearn_pred):.4f}")
print(f"Mean Absolute Probability Difference: {np.mean(np.abs(our_proba - sklearn_proba)):.6f}")

In [ ]:
# Compare prediction times
import time

# Generate larger dataset for timing
n_test_samples = 1000
X_large = np.random.randn(n_test_samples, X_train_scaled.shape[1])

# Time our implementation
start = time.time()
_ = our_knn.predict(X_large)
our_time = time.time() - start

# Time sklearn (with KD-tree optimization)
start = time.time()
_ = sklearn_knn.predict(X_large)
sklearn_time = time.time() - start

print(f"\nPrediction Time for {n_test_samples} samples:")
print(f"  Our Implementation: {our_time:.4f} seconds")
print(f"  Sklearn (with KD-tree): {sklearn_time:.4f} seconds")
print(f"  Sklearn is {our_time/sklearn_time:.1f}x faster")
print(f"\nNote: sklearn uses KD-tree or Ball tree for efficient nearest neighbor search")

In [ ]:
# Compare decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Our implementation
our_knn_2d = KNNClassifier(n_neighbors=5)
our_knn_2d.fit(X_2d, y_2d)
plot_decision_boundary_knn(X_2d, y_2d, our_knn_2d, 
                          'Our Implementation (K=5)', ax=axes[0])

# Sklearn - need to wrap for our plotting function
class SklearnKNNWrapper:
    def __init__(self, model):
        self.model = model
    def predict(self, X):
        return self.model.predict(X)

sklearn_knn_2d = KNeighborsClassifier(n_neighbors=5)
sklearn_knn_2d.fit(X_2d, y_2d)
sklearn_wrapper = SklearnKNNWrapper(sklearn_knn_2d)
plot_decision_boundary_knn(X_2d, y_2d, sklearn_wrapper, 
                          'Sklearn Implementation (K=5)', ax=axes[1])

plt.suptitle('Decision Boundary Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Feature comparison table
print("\nFeature Comparison: Our Implementation vs Sklearn")
print("=" * 70)
features = [
    ("Distance Metrics", "Euclidean, Manhattan, Minkowski", "Many more (Hamming, Cosine, etc.)"),
    ("Weight Functions", "Uniform, Distance", "Uniform, Distance, Custom callable"),
    ("Algorithm", "Brute force", "Auto (KD-tree, Ball tree, Brute)"),
    ("Multi-output", "No", "Yes"),
    ("Radius Neighbors", "No", "Yes"),
    ("Parallel Processing", "No", "Yes (n_jobs parameter)"),
    ("GPU Support", "No", "No (use cuML for GPU)"),
]

print(f"{'Feature':<25} {'Our Implementation':<25} {'Sklearn':<25}")
print("-" * 70)
for feature, ours, sklearn in features:
    print(f"{feature:<25} {ours:<25} {sklearn:<25}")

## Summary & Key Takeaways

### What We Learned:

1. **Algorithm Fundamentals**
   - KNN is a lazy learner - no explicit training phase
   - Predictions based on majority vote of K nearest neighbors
   - Distance metric choice affects decision boundaries

2. **Implementation Details**
   - Vectorized distance calculations for efficiency
   - Support for multiple distance metrics
   - Uniform vs distance-weighted voting

3. **Critical Considerations**
   - Feature scaling is essential
   - Optimal K found via cross-validation
   - Curse of dimensionality limits high-d applications

4. **Practical Guidelines**
   - Best for small datasets (n < 10,000)
   - Best for low dimensions (d < 20)
   - Use KD-trees for faster queries

### Key Insights:

- **K=1**: Memorizes training data (overfitting)
- **Large K**: Smoother boundaries (underfitting)
- **Distance weighting**: Reduces impact of outliers
- **Scaling**: Without it, high-range features dominate

### Next Steps:

- Implement KD-tree for O(log n) neighbor search
- Add support for regression (KNN Regressor)
- Implement radius-based neighbor search
- Explore approximate nearest neighbor methods (LSH, ANNOY)